# Course Scheduling Optimization — W27

This notebook follows the six steps of the course tutorial. The logic lives in the `coursesched`
package (`src/coursesched/`) so it can be unit-tested and re-run for sensitivity analysis; each cell
below calls the function for its step and shows the result.

| Step | What happens | Code |
|---|---|---|
| 1 | Read, clean and validate the workbook | `loader.load_workbook` |
| 2 | Build sets and parameters ($P_c$, soft cases, $\mathcal{I}$, $\mathcal{H}$, $\mathcal{G}_j$, $\Omega_{cc'}$) | `preprocess.build_model_data` |
| 3–4 | Decision variables, constraints C1–C7, objective | `model.build_model` |
| 5 | Solve with Gurobi | `model.solve` |
| 6 | Independent audit, tables, plots, export | `checker`, `report`, `viz` |

The full formulation is in `docs/formulation.md`; data findings are in `docs/data_notes.md`.

**Inputs:** put the course workbooks in `data/raw/` (they are not committed). The W27 model has
~2,160 variables and ~17,900 constraints, so it needs a full Gurobi license (the free academic
license works); the pip-bundled license is limited to 2,000 variables/constraints.

In [ ]:
import os
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import display

from coursesched import (
    build_model,
    build_model_data,
    check_schedule,
    diagnose_infeasibility,
    export_results,
    load_workbook,
    meetings_from_assignment,
    solve,
)
from coursesched.report import group_overlap_table, nonprime_table, schedule_table, soft_request_table
from coursesched.viz import plot_weekly

# Change this to the workbook you want to schedule (or set the COURSESCHED_INPUT variable).
INPUT_PATH = Path(os.environ.get("COURSESCHED_INPUT", ROOT / "data" / "raw" / "scheduling_input_W27.xlsx"))
OUTPUT_DIR = ROOT / "results"
print("Input:", INPUT_PATH)

## Step 1 — Read data

`load_workbook` reads the required sheets (`Components`, `Requests`, `ConflictGroups`, `Settings`)
and the optional `FixedSchedule` and `CourseSummary` sheets, keeps only active rows, normalizes
values (times → minutes, days → canonical `MTWRF` order, co-taught instructors split on `/`),
and runs validation checks.

Cleaning decisions worth knowing:
- The settings key `Quarter` (validation workbook) is treated as `term` (W27 workbook).
- Each active `FixedSchedule` row becomes a hard `UNAVAILABLE` request for every member of its target
  conflict group.
- Whether a request is hard comes from its `hard` column, not its name (W27 has hard `PREFER_*` requests).

In [ ]:
inputs = load_workbook(INPUT_PATH)
print(f"Term {inputs.term}: {len(inputs.components)} active components, {len(inputs.requests)} active requests, "
      f"{len(inputs.group_members)} conflict groups")

validation = pd.DataFrame(
    [(check, "error" if check in inputs.errors() else "warning", len(items), items)
     for check, items in inputs.validation.items()],
    columns=["check", "severity", "problems", "details"],
)
display(validation)

In [ ]:
components_df = pd.DataFrame([asdict(c) for c in inputs.components.values()])
requests_df = pd.DataFrame([asdict(r) for r in inputs.requests])
display(components_df.head(10))
display(requests_df)
display(pd.Series(inputs.settings, name="value").to_frame())

## Step 2 — Process and combine data

`build_model_data` generates every meeting pattern, keeps for each component only the patterns of its
family that satisfy all hard requests ($P_c$), records soft-request cases with their violating
patterns ($B_v$), forms the instructor / parent–child / conflict-group pairs, and precomputes the local
overlap sets $\Omega_{cc'}$ that become the C2, C3 and C5 constraint indices.

It stops early with a clear message if a hard request leaves a component with no pattern, or if a
department's non-prime quota cannot possibly be met.

In [ ]:
data = build_model_data(inputs)
display(pd.Series(data.summary(), name="count").to_frame())

patterns_df = pd.DataFrame([asdict(p) for p in data.patterns.values()])
display(patterns_df.groupby("family").agg(patterns=("id", "size"), nonprime=("nonprime", "sum")))

In [ ]:
# Components fixed to a single pattern by hard requests
display(pd.DataFrame(
    [(c, data.patterns[p].label, list(data.hard_removals[c])) for c, p in data.pinned_components().items()],
    columns=["component", "pinned to", "by requests"],
))

# Non-prime quotas: required count vs. components that could be non-prime at all
display(pd.DataFrame(
    [(dept, size, len(cids), data.np_required[(dept, size)],
      sum(any(data.patterns[p].nonprime for p in data.P[c]) for c in cids))
     for (dept, size), cids in data.np_sets.items()],
    columns=["department", "size class", "components", "required non-prime", "could be non-prime"],
))

## Steps 3–4 — Decision variables, constraints and objective

$x_{cp}\in\{0,1\}$ assigns pattern $p\in P_c$ to component $c$; $y_v\in\{0,1\}$ flags a violated soft
request case $v$; $o_{jcc'}\in\{0,1\}$ flags an overlap of $c,c'$ inside conflict group $j$.

$$
\begin{aligned}
\min\ & W_O \sum_j w_j \sum_{\{c,c'\}\in\mathcal{G}_j} o_{jcc'} \;+\; W_V \sum_{v\in V} w_v\, y_v \\
\text{s.t. }
& \textstyle\sum_{p\in P_c} x_{cp} = 1 && \forall c && \text{(C1)}\\
& x_{cp} + x_{c'p'} \le 1 && \forall \{c,c'\}\in\mathcal{I},\ (p,p')\in\Omega_{cc'} && \text{(C2)}\\
& x_{cp} + x_{c'p'} \le 1 && \forall (c,c')\in\mathcal{H},\ (p,p')\in\Omega_{cc'} && \text{(C3)}\\
& y_v \ge x_{c(v)p} && \forall v\in V,\ p\in B_v && \text{(C4)}\\
& x_{cp} + x_{c'p'} - o_{jcc'} \le 1 && \forall j,\ \{c,c'\}\in\mathcal{G}_j,\ (p,p')\in\Omega_{cc'} && \text{(C5)}\\
& \textstyle\sum_{c\in C_d^R}\sum_{p\in P_c^{NP}} x_{cp} \ge \lceil 0.35\,|C_d^R|\rceil && \forall d && \text{(C6)}\\
& \textstyle\sum_{c\in C_d^L}\sum_{p\in P_c^{NP}} x_{cp} \ge \lceil 0.30\,|C_d^L|\rceil && \forall d && \text{(C7)}
\end{aligned}
$$

Variables are only created where they can matter: $y_v$ only if $B_v\neq\emptyset$, $o_{jcc'}$ only if
$\Omega_{cc'}\neq\emptyset$.

In [ ]:
sm = build_model(data)
m = sm.model
print(f"Variables: {m.NumVars} ({m.NumBinVars} binary)   Constraints: {m.NumConstrs}")
print(f"Weights: W_O = {sm.overlap_weight:g}, W_V = {sm.request_weight:g}")

## Step 5 — Solve

Time limit and MIP gap default to the workbook's `Settings` sheet; pass `time_limit=`/`mip_gap=` to override.

In [ ]:
solution = solve(sm)
print(f"Status: {solution.status}   objective: {solution.objective}   bound: {solution.bound}   "
      f"gap: {solution.gap}   time: {solution.runtime:.2f}s")
if not solution.has_solution:
    if solution.status in {"INFEASIBLE", "INF_OR_UNBD"}:
        print("Conflicting constraints (IIS):", diagnose_infeasibility(sm))
    raise RuntimeError("No feasible schedule found. Check the input and hard constraints.")

## Step 6 — Analyze and export results

**Feasibility.** The checker re-verifies every hard rule from the finished schedule alone — it does not
use the model or the precomputed overlap sets and it detects overlaps minute by minute — and recomputes
the objective. Agreement with the solver is strong evidence that preprocessing and model are right.

In [ ]:
check = check_schedule(inputs, meetings_from_assignment(data, solution.assignment),
                       overlap_weight=sm.overlap_weight, request_weight=sm.request_weight)
print(check.summary())
assert check.feasible, "schedule breaks a hard rule"
assert abs(check.objective - solution.objective) < 1e-6, "checker and solver disagree on the objective"


In [ ]:
schedule = schedule_table(data, solution.assignment)
display(schedule[["component_id", "instructor", "department", "enrollment", "days", "start_time",
                  "end_time", "nonprime", "feasible_patterns"]])

In [ ]:
print("Non-prime quotas")
display(nonprime_table(check))

overlaps = group_overlap_table(data, solution.assignment)
print(f"Conflict-group pairs overlapping: {overlaps['overlap'].sum()} of {len(overlaps)} "
      f"({overlaps['unavoidable'].sum()} unavoidable given the hard rules)")
display(overlaps[overlaps["overlap"]])

print("Soft requests")
display(soft_request_table(data, solution.assignment))

In [ ]:
fig = plot_weekly(schedule, prime_window=(inputs.setting_minutes("prime_start"),
                                          inputs.setting_minutes("prime_end")))

In [ ]:
path = export_results(OUTPUT_DIR, data, solution, check)
fig.savefig(OUTPUT_DIR / f"{inputs.source.stem}_weekly.png", dpi=150)
print("Wrote", path)

## Next: deeper analysis

Planned analyses (see `README.md`):
- **Ties.** With the default settings every overlap and soft request can be avoided, so many schedules
  share the optimal objective. Compare them (Gurobi solution pool) and add a secondary objective
  (e.g. avoid late evenings, avoid surplus non-prime meetings) to choose among them.
- **Sensitivity.** Sweep the non-prime rates, the prime window and the large-enrollment threshold;
  find where the problem becomes infeasible or starts forcing overlaps.
- **Cost of each hard request.** Relax one at a time, re-solve, report the change.
- **Trade-offs.** Sweep $W_O/W_V$ once scenarios exist where overlaps and requests compete.